# Building a Local Data Lake

**BINFX410 — Chapter 10, Notebook 2 of 5**

---

In this notebook we build a complete data lake on your local filesystem using **pandas** and **PyArrow**. We implement the medallion architecture (Bronze → Silver → Gold), query Parquet files with DuckDB, and deliberately reproduce two classic data lake failure modes: **schema drift** and the **data swamp**.

**Prerequisites:** Run `01_introduction_and_setup.ipynb` first to generate `./raw_data/`.

## Learning Objectives

1. Explain the medallion (Bronze/Silver/Gold) architecture.
2. Convert CSV data to Parquet using PyArrow with compression.
3. Apply data quality transformations in the Silver layer.
4. Build pre-aggregated Gold tables for analytics.
5. Query Parquet files directly with DuckDB.
6. Understand schema drift and the data swamp problem.

## 1. Data Lake Concepts

### 1.1 Schema-on-Read vs Schema-on-Write

```
SCHEMA-ON-WRITE (Warehouse)          SCHEMA-ON-READ (Lake)
────────────────────────────         ─────────────────────
Data → [Validate] → Store            Data → Store → [Validate at query time]
  - Schema enforced at ingest          - Store anything immediately
  - Bad data rejected early            - Flexible, but risky
  - Query time is fast                 - Query must handle messy data
  - Changing schema is painful         - Schema evolves freely
```

### 1.2 The Medallion (Zone) Architecture

```
  RAW DATA                BRONZE                  SILVER                  GOLD
  ────────                ──────                  ──────                  ────
  CSV / VCF /             Parquet                 Cleaned                 Aggregated
  FASTQ / BAM   ──────►   (as-is conversion)  ►  (validated,         ►   (ready for
                          Immutable               typed, deduped)         dashboards)
                          Never delete
```

| Zone | Purpose | Changes allowed? | Format |
|------|---------|-----------------|--------|
| **Bronze (Raw)** | Faithful copy of source | Never (append only) | Parquet (from any source) |
| **Silver (Cleaned)** | Validated, typed, joined | Allowed with versioning | Parquet |
| **Gold (Curated)** | Business-ready aggregates | Rebuilt on schedule | Parquet |

### 1.3 Why Parquet?

Parquet is a **columnar** binary format designed for analytics:

```
CSV (row-oriented)               Parquet (column-oriented)
──────────────────               ─────────────────────────
id, gene,  af_tumor              id:       [1, 2, 3, ...]
1,  TP53,   0.45     ──────►     gene:     [TP53, BRCA1, ...]
2,  BRCA1,  0.32                 af_tumor: [0.45, 0.32, ...]
3,  KRAS,   0.78

Benefit: SELECT af_tumor FROM ... reads ONLY the af_tumor column bytes
```

Parquet also supports **predicate pushdown**: filters are applied at the file level using row group statistics, avoiding reading irrelevant data.

## 2. Setup

In [1]:
import os
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import duckdb

# Create lake directory structure
for zone in ['bronze', 'silver', 'gold']:
    os.makedirs(f'./data_lake/{zone}', exist_ok=True)

print('PyArrow version:', pa.__version__)
print('DuckDB version: ', duckdb.__version__)
print('Directory structure:')
for zone in ['bronze', 'silver', 'gold']:
    print(f'  ./data_lake/{zone}/')

PyArrow version: 21.0.0
DuckDB version:  1.4.3
Directory structure:
  ./data_lake/bronze/
  ./data_lake/silver/
  ./data_lake/gold/


## 3. Bronze Layer — Raw Ingestion

The Bronze layer makes a faithful, **compressed copy** of every source file. We convert CSV → Parquet but apply zero business logic. This layer is **append-only** and **never deleted** — it is your audit trail.

In [2]:
# Read all CSV source files
tables = {
    'samples':       pd.read_csv('./raw_data/samples.csv'),
    'genes':         pd.read_csv('./raw_data/genes.csv'),
    'variant_calls': pd.read_csv('./raw_data/variant_calls.csv'),
    'variants':      pd.read_csv('./raw_data/variants.csv'),
}

for name, df in tables.items():
    print(f'  {name:15s}: {len(df):,} rows, dtypes: {dict(df.dtypes)}')

  samples        : 500 rows, dtypes: {'sample_id': dtype('int64'), 'patient_id': dtype('int64'), 'tissue_type': dtype('O'), 'sequencing_platform': dtype('O'), 'library_prep': dtype('O'), 'collection_date': dtype('O'), 'mean_coverage': dtype('float64'), 'pct_bases_20x': dtype('float64'), 'project_id': dtype('O')}
  genes          : 100 rows, dtypes: {'gene_id': dtype('int64'), 'gene_name': dtype('O'), 'chromosome': dtype('O'), 'start_pos': dtype('int64'), 'end_pos': dtype('int64'), 'strand': dtype('O'), 'biotype': dtype('O'), 'is_cancer_gene': dtype('bool')}
  variant_calls  : 2,000 rows, dtypes: {'call_id': dtype('int64'), 'sample_id': dtype('int64'), 'call_date': dtype('O'), 'caller': dtype('O'), 'filter_status': dtype('O')}
  variants       : 5,000 rows, dtypes: {'variant_id': dtype('int64'), 'call_id': dtype('int64'), 'gene_id': dtype('int64'), 'chromosome': dtype('O'), 'position': dtype('int64'), 'ref_allele': dtype('O'), 'alt_allele': dtype('O'), 'variant_type': dtype('O'), 'conse

In [3]:
# Write to Bronze as Parquet with Snappy compression
# We use PyArrow directly for fine-grained control over schema and compression

csv_sizes  = {}
pq_sizes   = {}

for name, df in tables.items():
    src_path  = f'./raw_data/{name}.csv'
    dest_path = f'./data_lake/bronze/{name}.parquet'

    arrow_table = pa.Table.from_pandas(df)
    pq.write_table(arrow_table, dest_path, compression='snappy')

    csv_sizes[name] = os.path.getsize(src_path)
    pq_sizes[name]  = os.path.getsize(dest_path)

print(f'{"Table":<18} {"CSV (KB)":>10} {"Parquet (KB)":>14} {"Compression":>12}')
print('-' * 58)
for name in tables:
    csv_kb = csv_sizes[name] / 1024
    pq_kb  = pq_sizes[name]  / 1024
    ratio  = csv_sizes[name] / pq_sizes[name]
    print(f'{name:<18} {csv_kb:>10.1f} {pq_kb:>14.1f} {ratio:>11.2f}x')

Table                CSV (KB)   Parquet (KB)  Compression
----------------------------------------------------------
samples                  33.9           19.6        1.73x
genes                     5.2            7.9        0.66x
variant_calls            70.0           27.0        2.59x
variants                347.6          152.4        2.28x


In [4]:
# Inspect the Parquet schema — PyArrow infers types from the DataFrame
pf = pq.ParquetFile('./data_lake/bronze/samples.parquet')
print('=== samples.parquet schema ===')
print(pf.schema_arrow)
print()
print(f'Row groups : {pf.metadata.num_row_groups}')
print(f'Total rows : {pf.metadata.num_rows:,}')

=== samples.parquet schema ===
sample_id: int64
patient_id: int64
tissue_type: string
sequencing_platform: string
library_prep: string
collection_date: string
mean_coverage: double
pct_bases_20x: double
project_id: string
-- schema metadata --
pandas: '{"index_columns": [{"kind": "range", "name": null, "start": 0, "' + 1381

Row groups : 1
Total rows : 500


## 4. Silver Layer — Cleaning and Typing

In the Silver layer we apply business rules:

- Cast string dates to proper `datetime64` types
- Validate foreign key integrity
- Standardize text casing
- Drop or impute nulls
- Add derived columns (e.g., `signal_strength = depth * af_tumor`)

Silver is the layer most analysts query day-to-day.

In [5]:
# Read bronze
df_samples_b = pd.read_parquet('./data_lake/bronze/samples.parquet')
df_genes_b   = pd.read_parquet('./data_lake/bronze/genes.parquet')
df_calls_b   = pd.read_parquet('./data_lake/bronze/variant_calls.parquet')
df_vars_b    = pd.read_parquet('./data_lake/bronze/variants.parquet')

print('Bronze types (before cleaning):')
print(df_samples_b.dtypes)

Bronze types (before cleaning):
sample_id                int64
patient_id               int64
tissue_type             object
sequencing_platform     object
library_prep            object
collection_date         object
mean_coverage          float64
pct_bases_20x          float64
project_id              object
dtype: object


In [6]:
# ── Silver: Samples ────────────────────────────────────────────────────────────
df_samples_s = df_samples_b.copy()

# Cast date strings to actual datetime
df_samples_s['collection_date'] = pd.to_datetime(df_samples_s['collection_date'])

# Standardize tissue_type to title case
df_samples_s['tissue_type'] = df_samples_s['tissue_type'].str.strip().str.title()

# Validate coverage is positive
before = len(df_samples_s)
df_samples_s = df_samples_s[df_samples_s['mean_coverage'] > 0]
print(f'Dropped {before - len(df_samples_s)} samples with non-positive coverage')

# Remove duplicate sample IDs
before = len(df_samples_s)
df_samples_s = df_samples_s.drop_duplicates(subset=['sample_id'])
print(f'Duplicates removed from samples: {before - len(df_samples_s)}')

pq.write_table(pa.Table.from_pandas(df_samples_s), './data_lake/silver/samples.parquet', compression='snappy')
print('Silver samples written:', len(df_samples_s), 'rows')
print(df_samples_s.dtypes)

Dropped 0 samples with non-positive coverage
Duplicates removed from samples: 0
Silver samples written: 500 rows
sample_id                       int64
patient_id                      int64
tissue_type                    object
sequencing_platform            object
library_prep                   object
collection_date        datetime64[ns]
mean_coverage                 float64
pct_bases_20x                 float64
project_id                     object
dtype: object


In [7]:
# ── Silver: Genes ─────────────────────────────────────────────────────────────
df_genes_s = df_genes_b.copy()

# Ensure start_pos < end_pos (swap if needed)
mask = df_genes_s['start_pos'] >= df_genes_s['end_pos']
if mask.any():
    print(f'Swapping start/end for {mask.sum()} genes where start >= end')
    df_genes_s.loc[mask, ['start_pos', 'end_pos']] = (
        df_genes_s.loc[mask, ['end_pos', 'start_pos']].values
    )

# Cast is_cancer_gene to bool
df_genes_s['is_cancer_gene'] = df_genes_s['is_cancer_gene'].astype(bool)

pq.write_table(pa.Table.from_pandas(df_genes_s), './data_lake/silver/genes.parquet', compression='snappy')
print('Silver genes written:', len(df_genes_s), 'rows')
df_genes_s[['gene_id', 'gene_name', 'chromosome', 'biotype', 'is_cancer_gene']].head(3)

Silver genes written: 100 rows


,gene_id,gene_name,chromosome,biotype,is_cancer_gene
0,1,TP53,chr1,protein_coding,True
1,2,BRCA1,chr1,protein_coding,True
2,3,BRCA2,chrX,pseudogene,True


In [8]:
# ── Silver: Variant Calls ─────────────────────────────────────────────────────
df_calls_s = df_calls_b.copy()
df_calls_s['call_date'] = pd.to_datetime(df_calls_s['call_date'])

# Validate FK: every call must have a valid sample
valid_samples   = set(df_samples_s['sample_id'])
orphan_calls    = df_calls_s[~df_calls_s['sample_id'].isin(valid_samples)]
print(f'Orphan calls (no matching sample): {len(orphan_calls)}')
df_calls_s = df_calls_s[df_calls_s['sample_id'].isin(valid_samples)]

# Add year/month columns for easy partitioning
df_calls_s['call_year']  = df_calls_s['call_date'].dt.year
df_calls_s['call_month'] = df_calls_s['call_date'].dt.month

pq.write_table(pa.Table.from_pandas(df_calls_s), './data_lake/silver/variant_calls.parquet', compression='snappy')
print('Silver variant_calls written:', len(df_calls_s), 'rows')

Orphan calls (no matching sample): 0
Silver variant_calls written: 2000 rows


In [9]:
# ── Silver: Variants ──────────────────────────────────────────────────────────
df_vars_s = df_vars_b.copy()

# Validate ref != alt
before = len(df_vars_s)
df_vars_s = df_vars_s[df_vars_s['ref_allele'] != df_vars_s['alt_allele']]
print(f'Dropped {before - len(df_vars_s)} variants where ref == alt')

# Remove rows with very low depth (quality filter)
before = len(df_vars_s)
df_vars_s = df_vars_s[df_vars_s['depth'] >= 10]
print(f'Dropped {before - len(df_vars_s)} variants with depth < 10')

# Add signal_strength = depth * af_tumor (analogous to line_revenue)
df_vars_s['signal_strength'] = (df_vars_s['depth'] * df_vars_s['af_tumor']).round(2)

pq.write_table(
    pa.Table.from_pandas(df_vars_s, preserve_index=False),
    './data_lake/silver/variants.parquet',
    compression='snappy'
)
print('Silver variants written:', len(df_vars_s), 'rows')
df_vars_s.head(3)

Dropped 0 variants where ref == alt
Dropped 0 variants with depth < 10
Silver variants written: 5000 rows


,variant_id,call_id,gene_id,chromosome,position,ref_allele,alt_allele,variant_type,consequence,af_tumor,depth,quality_score,signal_strength
0,1,388,52,chr10,121745068,C,G,Deletion,frameshift_variant,0.5168,75,47.25,38.76
1,2,1324,32,chr20,14654760,T,A,SNV,missense_variant,0.5677,142,22.63,80.61
2,3,207,36,chr16,120483867,G,T,Insertion,missense_variant,0.2479,441,55.18,109.32


## 5. Gold Layer — Analytics-Ready Aggregates

Gold tables are **pre-aggregated** for common analytical questions. They are rebuilt on a schedule (daily/hourly) from Silver. Analysts and BI tools query Gold directly for maximum performance.

In [10]:
# Read silver layers for Gold computation
df_samples_s  = pd.read_parquet('./data_lake/silver/samples.parquet')
df_genes_s    = pd.read_parquet('./data_lake/silver/genes.parquet')
df_calls_s    = pd.read_parquet('./data_lake/silver/variant_calls.parquet')
df_vars_s     = pd.read_parquet('./data_lake/silver/variants.parquet')

print('Silver tables loaded.')

Silver tables loaded.


In [11]:
# ── Gold: Monthly Variant Counts ──────────────────────────────────────────────
# Join variants → variant_calls to get call_date on each variant
vars_with_call = df_vars_s.merge(
    df_calls_s[['call_id', 'call_date', 'call_year', 'call_month', 'filter_status', 'sample_id']],
    on='call_id', how='inner'
)

# PASS calls only
pass_vars = vars_with_call[vars_with_call['filter_status'] == 'PASS']

monthly_variant_counts = (
    pass_vars
    .groupby(['call_year', 'call_month'])
    .agg(
        variant_count=('variant_id', 'count'),
        unique_samples=('sample_id', 'nunique'),
        mean_af_tumor=('af_tumor', 'mean'),
    )
    .reset_index()
    .assign(mean_af_tumor=lambda x: x['mean_af_tumor'].round(4))
    .sort_values(['call_year', 'call_month'])
)

pq.write_table(pa.Table.from_pandas(monthly_variant_counts), './data_lake/gold/monthly_variant_counts.parquet', compression='snappy')
print('Gold: monthly_variant_counts —', len(monthly_variant_counts), 'rows')
monthly_variant_counts.tail(6)

Gold: monthly_variant_counts — 59 rows


,call_year,call_month,variant_count,unique_samples,mean_af_tumor
53,2024,7,34,12,0.5007
54,2024,8,18,7,0.4564
55,2024,9,18,7,0.5869
56,2024,10,7,3,0.4672
57,2024,11,7,3,0.4960
58,2024,12,1,1,0.6409


In [12]:
# ── Gold: Gene Hotspots ───────────────────────────────────────────────────────
vars_pass = pass_vars.merge(
    df_genes_s[['gene_id', 'gene_name', 'biotype', 'is_cancer_gene']],
    on='gene_id', how='left'
)

gene_hotspots = (
    vars_pass
    .groupby(['gene_id', 'gene_name', 'biotype', 'is_cancer_gene'])
    .agg(
        total_variants=('variant_id', 'count'),
        unique_samples=('sample_id', 'nunique'),
        mean_af_tumor=('af_tumor', 'mean'),
        most_common_consequence=('consequence', lambda x: x.mode()[0] if len(x) > 0 else None),
    )
    .reset_index()
    .assign(mean_af_tumor=lambda x: x['mean_af_tumor'].round(4))
    .sort_values('total_variants', ascending=False)
)

pq.write_table(pa.Table.from_pandas(gene_hotspots), './data_lake/gold/gene_hotspots.parquet', compression='snappy')
print('Gold: gene_hotspots —', len(gene_hotspots), 'rows')
gene_hotspots.head(5)

Gold: gene_hotspots — 100 rows


,gene_id,gene_name,biotype,is_cancer_gene,total_variants,unique_samples,mean_af_tumor,most_common_consequence
78,79,JAK3,pseudogene,False,34,34,0.5019,frameshift_variant
35,36,RAS4,miRNA,False,33,29,0.5133,intron_variant
82,83,SRC3,protein_coding,True,33,30,0.4764,intron_variant
97,98,FOXO2,protein_coding,True,32,31,0.4854,missense_variant
99,100,FOXO4,lncRNA,False,31,30,0.4543,splice_site_variant


In [13]:
# ── Gold: Sample Mutation Burden ──────────────────────────────────────────────
sample_mutation_burden = (
    pass_vars
    .groupby('sample_id')
    .agg(
        total_variants=('variant_id', 'count'),
        mean_depth=('depth', 'mean'),
        mean_af_tumor=('af_tumor', 'mean'),
    )
    .reset_index()
    .merge(
        df_samples_s[['sample_id', 'patient_id', 'tissue_type', 'project_id', 'collection_date']],
        on='sample_id'
    )
    .assign(
        mean_depth=lambda x: x['mean_depth'].round(1),
        mean_af_tumor=lambda x: x['mean_af_tumor'].round(4),
    )
    .sort_values('total_variants', ascending=False)
)

pq.write_table(pa.Table.from_pandas(sample_mutation_burden), './data_lake/gold/sample_mutation_burden.parquet', compression='snappy')
print('Gold: sample_mutation_burden —', len(sample_mutation_burden), 'rows')
sample_mutation_burden.head(5)

Gold: sample_mutation_burden — 416 rows


,sample_id,total_variants,mean_depth,mean_af_tumor,patient_id,tissue_type,project_id,collection_date
7,9,23,260.7,0.4532,190,Blood,PROJ_002,2021-09-08
347,413,19,341.9,0.5419,259,Normal,PROJ_003,2022-09-09
88,104,19,282.0,0.5076,218,Tumor,PROJ_002,2022-02-22
247,291,19,280.2,0.5477,179,Tumor,PROJ_004,2020-01-06
412,497,18,243.2,0.5724,10,Tumor,PROJ_003,2021-08-13


## 6. Querying the Data Lake with DuckDB

DuckDB can query Parquet files **directly on disk** without loading them into memory. This is extremely powerful for data lake analytics — you get SQL semantics on raw files.

In [14]:
import duckdb

# DuckDB can query Parquet files directly using read_parquet() or glob patterns
con = duckdb.connect()  # in-memory connection

# Query 1: Top 5 biotypes by variant count (PASS calls only)
result = con.execute("""
    SELECT
        g.biotype,
        COUNT(v.variant_id)   AS variant_count,
        COUNT(DISTINCT v.call_id) AS unique_calls
    FROM read_parquet('./data_lake/silver/variants.parquet')      v
    JOIN read_parquet('./data_lake/silver/genes.parquet')         g
      ON v.gene_id = g.gene_id
    JOIN read_parquet('./data_lake/silver/variant_calls.parquet') c
      ON v.call_id = c.call_id
    WHERE c.filter_status = 'PASS'
    GROUP BY g.biotype
    ORDER BY variant_count DESC
""").df()

print('=== Top Biotypes by Variant Count (PASS only) ===')
print(result)

=== Top Biotypes by Variant Count (PASS only) ===
          biotype  variant_count  unique_calls
0  protein_coding           1372           754
1          lncRNA            632           478
2      pseudogene            241           220
3           miRNA            162           146


In [15]:
# Query 2: Monthly variant trend using the Gold table
trend = con.execute("""
    SELECT
        call_year,
        call_month,
        variant_count,
        unique_samples,
        ROUND(mean_af_tumor, 4) AS mean_af_tumor
    FROM read_parquet('./data_lake/gold/monthly_variant_counts.parquet')
    ORDER BY call_year, call_month
    LIMIT 12
""").df()

print('=== Monthly Variant Counts (first 12 months) ===')
print(trend.to_string(index=False))

=== Monthly Variant Counts (first 12 months) ===
 call_year  call_month  variant_count  unique_samples  mean_af_tumor
      2020           1              4               2         0.7627
      2020           3             10               4         0.6334
      2020           4             10               3         0.3773
      2020           5             13               5         0.5590
      2020           6             18               6         0.5063
      2020           7             19               7         0.4152
      2020           8             18               6         0.4788
      2020           9             38              13         0.4990
      2020          10             39              18         0.4586
      2020          11             64              19         0.5116
      2020          12             49              18         0.4827
      2021           1             41              15         0.4744


In [16]:
# Query 3: Read ALL parquet files in the Gold directory using glob
all_gold = con.execute("""
    SELECT
        filename,
        COUNT(*) AS row_count
    FROM read_parquet('./data_lake/gold/*.parquet', filename=true)
    GROUP BY filename
""").df()
print('=== Gold Layer Files ===')
print(all_gold)

=== Gold Layer Files ===
                                          filename  row_count
0  ./data_lake/gold/monthly_variant_counts.parquet         59
1           ./data_lake/gold/gene_hotspots.parquet        100
2  ./data_lake/gold/sample_mutation_burden.parquet        416


## 7. Schema Drift — The Hidden Data Lake Problem

One of the biggest risks in a data lake is **schema drift**: when the structure of incoming data changes, older files become incompatible with newer ones. Without enforcement, queries silently return wrong results or crash.

Let's reproduce this problem deliberately.

In [17]:
os.makedirs('./data_lake/demo_drift', exist_ok=True)

# Write "old" batch of variants (original schema — no quality_score or depth)
df_vars_s_loaded = pd.read_parquet('./data_lake/silver/variants.parquet')
df_batch1 = df_vars_s_loaded[['variant_id', 'gene_id', 'chromosome', 'position',
                               'ref_allele', 'alt_allele', 'af_tumor']].head(100)
pq.write_table(pa.Table.from_pandas(df_batch1), './data_lake/demo_drift/variants_batch1.parquet')

# New batch adds quality_score and depth (upgraded pipeline)
df_batch2 = df_vars_s_loaded[['variant_id', 'gene_id', 'chromosome', 'position',
                               'ref_allele', 'alt_allele', 'af_tumor',
                               'quality_score', 'depth']].tail(100).copy()
pq.write_table(pa.Table.from_pandas(df_batch2), './data_lake/demo_drift/variants_batch2.parquet')

print('Old schema (batch1):', pq.read_schema('./data_lake/demo_drift/variants_batch1.parquet'))
print()
print('New schema (batch2):', pq.read_schema('./data_lake/demo_drift/variants_batch2.parquet'))

Old schema (batch1): variant_id: int64
gene_id: int64
chromosome: string
position: int64
ref_allele: string
alt_allele: string
af_tumor: double
-- schema metadata --
pandas: '{"index_columns": [{"kind": "range", "name": null, "start": 0, "' + 1079

New schema (batch2): variant_id: int64
gene_id: int64
chromosome: string
position: int64
ref_allele: string
alt_allele: string
af_tumor: double
quality_score: double
depth: int64
-- schema metadata --
pandas: '{"index_columns": [{"kind": "range", "name": null, "start": 4900' + 1317


In [18]:
# DuckDB handles schema drift gracefully by unioning all columns,
# filling missing columns with NULL. Other tools may crash.

result = con.execute("""
    SELECT
        variant_id,
        chromosome,
        af_tumor,
        quality_score,  -- only exists in batch2
        depth           -- only exists in batch2
    FROM read_parquet('./data_lake/demo_drift/*.parquet', union_by_name=true)
    ORDER BY variant_id
    LIMIT 10
""").df()

print('=== Schema Drift Result (NULLs for missing columns) ===')
print(result)
print()
print('Rows with NULL quality_score (from old batch):', result['quality_score'].isna().sum())
print('Rows with NULL depth:', result['depth'].isna().sum())

=== Schema Drift Result (NULLs for missing columns) ===
   variant_id chromosome  af_tumor  quality_score  depth
0           1      chr10    0.5168            NaN   <NA>
1           2      chr20    0.5677            NaN   <NA>
2           3      chr16    0.2479            NaN   <NA>
3           4      chr22    0.3092            NaN   <NA>
4           5       chr2    0.2648            NaN   <NA>
5           6      chr16    0.4771            NaN   <NA>
6           7      chr11    0.3669            NaN   <NA>
7           8      chr20    0.6620            NaN   <NA>
8           9      chr20    0.7735            NaN   <NA>
9          10       chrX    0.3779            NaN   <NA>

Rows with NULL quality_score (from old batch): 10
Rows with NULL depth: 10


### What went wrong?

- Old batch files are missing `quality_score` and `depth` → those values are `NULL` for 100 rows.
- Any aggregate on `quality_score` or `depth` will be **silently wrong** — it excludes 100 records.
- Downstream consumers must now handle `NULL` defensively everywhere.

In a **lakehouse** (Notebook 04), Delta Lake's schema enforcement would have **rejected** the new batch file outright unless you explicitly ran schema evolution with `schema_mode='merge'`, ensuring all existing data gets a proper default value.

This is a major reason the lakehouse pattern was developed.

## 8. The Data Swamp — Governance Failure

A data lake without governance becomes a **data swamp**: files accumulate with no documentation, lineage, or discoverability. Let's simulate this.

In [19]:
import json
from datetime import datetime

os.makedirs('./data_lake/swamp_demo', exist_ok=True)

# Simulate a swamp: files with cryptic names, no metadata, mixed formats
files_dumped = [
    'variants_final_v2.vcf',
    'variants_final_FINAL.vcf',
    'sample_qc_march_backup.csv',
    'UNKNOWN_RUN_002.fastq.gz',
    'pipeline_output_DO_NOT_DELETE.tsv',
    'somatic_calls_2023_fixed.parquet',
    'somatic_calls_2023.parquet',
    'john_analysis_may.xlsx',
]

# Create dummy files
for fname in files_dumped:
    with open(f'./data_lake/swamp_demo/{fname}', 'w') as f:
        f.write('dummy data')

print('=== Data Swamp Contents ===')
for f in sorted(os.listdir('./data_lake/swamp_demo')):
    print(f'  {f}')
print()
print('Questions an analyst faces with no answers:')
print('  - Which VCF is the authoritative variant call set?')
print('  - Is variants_final_v2 or variants_final_FINAL more recent?')
print('  - What does UNKNOWN_RUN_002.fastq.gz contain?')
print('  - Was sample_qc_march_backup.csv modified by anyone?')
print('  - Are the two somatic_calls parquet files different?')

=== Data Swamp Contents ===
  UNKNOWN_RUN_002.fastq.gz
  john_analysis_may.xlsx
  pipeline_output_DO_NOT_DELETE.tsv
  sample_qc_march_backup.csv
  somatic_calls_2023.parquet
  somatic_calls_2023_fixed.parquet
  variants_final_FINAL.vcf
  variants_final_v2.vcf

Questions an analyst faces with no answers:
  - Which VCF is the authoritative variant call set?
  - Is variants_final_v2 or variants_final_FINAL more recent?
  - What does UNKNOWN_RUN_002.fastq.gz contain?
  - Was sample_qc_march_backup.csv modified by anyone?
  - Are the two somatic_calls parquet files different?


In [ ]:
# The solution: a data catalog / metadata manifest
# In a real lake this would be Apache Atlas, AWS Glue, or a data catalog tool.
# Here we show a minimal manifest pattern.

catalog = {
    "tables": [
        {
            "name": "samples",
            "zone": "silver",
            "path": "./data_lake/silver/samples.parquet",
            "owner": "bioinformatics_team",
            "description": "Cleaned sequencing sample manifest. Source: LIMS system.",
            "last_updated": datetime.utcnow().isoformat(),
            "schema": ["sample_id", "patient_id", "tissue_type", "sequencing_platform",
                       "library_prep", "collection_date", "mean_coverage", "pct_bases_20x", "project_id"],
            "row_count": 500,
            "sla": "refreshed after each sequencing run"
        }
    ]
}

with open('./data_lake/catalog.json', 'w') as f:
    json.dump(catalog, f, indent=2)

print('Minimal data catalog written to ./data_lake/catalog.json')
print(json.dumps(catalog, indent=2))

## 9. Data Lake Summary

In [ ]:
print('=== Data Lake Directory Summary ===')
for zone in ['bronze', 'silver', 'gold']:
    zone_path = f'./data_lake/{zone}'
    files = [f for f in os.listdir(zone_path) if f.endswith('.parquet')]
    total_bytes = sum(os.path.getsize(f'{zone_path}/{f}') for f in files)
    print(f'\n  {zone.upper()} ({zone_path}/)')
    for f in files:
        size = os.path.getsize(f'{zone_path}/{f}') / 1024
        rows = pq.read_metadata(f'{zone_path}/{f}').num_rows
        print(f'    {f:40s}  {size:6.1f} KB  {rows:>6,} rows')
    print(f'  Zone total: {total_bytes/1024:.1f} KB')

## Exercises

**Exercise 2.1 — DuckDB SQL on Parquet**

Using DuckDB and the Silver Parquet files, write a query that finds the **top 3 tissue types by total variant count** from PASS calls only. Your output should have columns: `tissue_type`, `variant_count`, `unique_samples`.

Hint: Join `variants` → `variant_calls` → `samples` to link variants to tissue type. Filter where `filter_status = 'PASS'`.

In [ ]:
# Exercise 2.1 — your query here
result_ex1 = con.execute("""
    -- TODO: write your query
    SELECT 'replace me' AS tissue_type, 0 AS variant_count, 0 AS unique_samples
""").df()
print(result_ex1)

**Exercise 2.2 — Build a New Gold Table**

Create a new Gold Parquet file `./data_lake/gold/consequence_monthly_counts.parquet` that shows:
- `consequence` (variant consequence type)
- `call_year`
- `call_month`
- `variant_count`
- `unique_samples`

Include only PASS calls. Sort by `consequence`, `call_year`, `call_month`.

In [ ]:
# Exercise 2.2 — build the gold table
# Use either pandas or DuckDB to compute the aggregation,
# then write to Parquet with PyArrow.

# df_consequence_monthly = ...
# pq.write_table(...)

**Exercise 2.3 — Schema Evolution Analysis**

The two files in `./data_lake/demo_drift/` have incompatible schemas (`variants_batch1.parquet` vs `variants_batch2.parquet`).

a) Write a Python function `check_schema_compatibility(path1, path2)` that returns `True` if both files have the same column names and types, `False` otherwise, and prints a diff of any differences.

b) How would you design a Bronze → Silver pipeline that detects schema changes automatically and alerts the bioinformatics team before writing to Silver?

In [ ]:
# Exercise 2.3a — schema compatibility checker
def check_schema_compatibility(path1: str, path2: str) -> bool:
    """Return True if schemas are identical, False otherwise."""
    # TODO: read schemas, compare, print diff
    pass

# Test it:
# check_schema_compatibility(
#     './data_lake/demo_drift/variants_batch1.parquet',
#     './data_lake/demo_drift/variants_batch2.parquet'
# )

---

**Next:** `03_data_warehouse.ipynb` — Building a star-schema data warehouse with DuckDB